In [1]:
import os
from dotenv import load_dotenv
from pdf_extractor import extract_text
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
MANUALS = {
    "iphone": {
        "label": "iPhone 15 Pro",
        "path": "iPhone_15_Pro_Manual.pdf",
    },
    "dell": {
        "label": "Dell Precision 5530",
        "path": "Dell_Precision_5530_Manual.pdf",
    },
}

In [4]:
MODEL = "gpt-4.1-mini"
openai = OpenAI()

SYSTEM_PROMPT = """You are a product manual assistant. You have access to the full text of exactly two manuals:
the iPhone 15 Pro manual and the DellPrecision 5530 manual, which are provided below.
 
Rules:
1. Only answer questions that can be answered using the manual content provided.
2. If the user's question is about a different product, or about anything not
   covered in these two manuals, respond that you don't have access to
   information about that product (or that topic) and can only help with the
   iPhone 15 Pro and Dell Precision 5530 manuals.
3. Do not use outside knowledge to answer questions about other products, even
   if you happen to know the answer.
4. When you do answer, base your answer strictly on the manual text given
   below, and mention which manual the answer came from.
5. Keep answers concise and directly useful.
 
--- IPHONE 15 PRO MANUAL ---
{iphone_text}
 
--- DELL PRECISION 5530 MANUAL ---
{dell_text}
"""

In [5]:
def load_manuals() -> dict:
    """Load and extract text for both manuals."""
    texts = {}
    for key, info in MANUALS.items():
        texts[key] = extract_text(info["path"])
        status = "loaded" if texts[key] else "MISSING"
        print(f"[{status}] {info['label']} ({info['path']})")
    return texts


def build_system_prompt(texts: dict) -> str:
    return SYSTEM_PROMPT.format(
        iphone_text=texts.get("iphone", "") or "(manual not available)",
        dell_text=texts.get("dell", "") or "(manual not available)",
    )

texts = load_manuals()
system_prompt = build_system_prompt(texts)

[loaded] iPhone 15 Pro (iPhone_15_Pro_Manual.pdf)
[loaded] Dell Precision 5530 (Dell_Precision_5530_Manual.pdf)


In [6]:
def messages_for(question: str) -> list:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

def ask(question: str) -> str:

    response = openai.chat.completions.create(
        model=MODEL,
        messages = messages_for(question)
    )

    return response.choices[0].message.content

In [7]:
question = "What is the regulatory model number for the Dell Precision 5530?"
answer = ask(question)
print(f"\nAssistant: {answer}\n")


Assistant: The regulatory model number for the Dell Precision 5530 is P56F. (From the Dell Precision 5530 manual)

